# X2 — Contexto em conversas longas

**Pergunta:** o que acontece quando a conversa fica longa demais para caber inteira no contexto?  
**Ideia central:** comprimir a conversa é perder informação. O que o código precisa saber (pedido, quanto já foi compensado) deve morar no **estado estruturado** (N1.5), não depender de a conversa sobreviver à compressão.

Notebook **opcional** e independente dos principais. Cada seção abre com `preparar_secao` e roda sozinha. Onde o modelo é roteirizado, a célula diz.

Tokens são estimados em 1 token ≈ 4 caracteres: serve para comparar estratégias, não para calcular custo.

> ▶ **Comece por aqui:** execute a célula de código abaixo **uma vez**, antes de qualquer seção. Ela importa tudo que os exemplos usam (`atender_caso`, `afirmar`, …). Depois disso, cada seção roda sozinha, na ordem que você quiser.

In [1]:
# Execute esta célula primeiro: ela importa o que todas as seções usam.
from rangoja import config, dados, falhas, ferramentas, controle, observabilidade, relogio
from rangoja.armazem import Armazem
from rangoja.casos import caso
from rangoja.execucao import atender, atender_caso, continuar, aprovar, recusar, expirar_pendencias, reiniciar_runner, estado_persistido
from rangoja.notebook import (preparar_ambiente, preparar_secao, mostrar, comparar, afirmar, esperar, verificar, verificar_falha,
                              resumo, amostra, matriz)
from rangoja.trajetoria import chamou, ferramenta_antes, contar_chamadas
from rangoja.versoes import v0, v1, v2, v3, v0_so_prompt, v1_agente, v2_controlado, v3_robusto, montar
preparar_ambiente()

from rangoja import extras
from rangoja.modelo.roteiro import chamada, texto

modo,replay
modelo,gpt-4o-mini
chave de API,não (use replay)
observabilidade,desligado (RANGOJA_PHOENIX=off)


## X2.1 — Uma conversa de 20 turnos

Parte do caso `C11` (pedido incompleto, em três falas). Primeiro o atendimento real, com o estado que o código guardou. Depois, uma conversa de 20 turnos **sintética** (escrita à mão), em que o pedido e as compensações aparecem só no começo e o resto é assunto solto.

In [2]:
s = preparar_secao("X2.1")
real = atender_caso(s.caso, com=s.correcao, replay=s.gravacoes["correcao"])
estado = {k: real.estado.get(k) for k in ("pedido_id", "compensacoes")}
print("estado estruturado do atendimento real:", estado)
for c in real.trajetoria:
    if c.efetiva and c.ferramenta in ("registrar_reembolso", "oferecer_cupom"):
        print(" ", c.ferramenta, c.args)

conversa = extras.conversa_longa(20)
completa = extras.texto_da_conversa(conversa)
print(f"\n{len(conversa)} turnos, ~{extras.tokens(completa)} tokens")
print(completa[:600], "...")

estado estruturado do atendimento real: {'pedido_id': '4822', 'compensacoes': 18.0}
  registrar_reembolso {'pedido_id': '4822', 'valor': 8, 'motivo': 'item_faltando'}
  oferecer_cupom {'pedido_id': '4822', 'valor': 10, 'motivo': 'atraso na entrega'}

20 turnos, ~332 tokens
cliente: Meu pedido chegou incompleto.
atendente: Sinto muito! Pode me dizer o número do pedido e o que faltou?
cliente: É o 4822, faltou o refri.
atendente: Consultei o pedido 4822. Reembolsei R$ 8,00 do refrigerante.
cliente: Ah, e faltou a sobremesa também.
atendente: Concedi um cupom de R$ 10,00 pela sobremesa que faltou. Total compensado: R$ 18,00.
cliente: Valeu! Aproveitando, o app tá lento aqui comigo.
atendente: Obrigado por avisar. Esse tipo de problema vai para a equipe do aplicativo.
cliente: Vocês entregam no bairro vizinho também?
atendente: A cobertura aparece no app, na tela de ...


## X2.2 — Janela deslizante

Mantém só os últimos *k* turnos. Barata e previsível, mas o que ficou fora da janela deixou de existir para o modelo.

In [3]:
janela = extras.texto_da_conversa(extras.janela_deslizante(conversa, 6))
print(janela)
print("\nfatos na janela:", extras.fatos_na_conversa(janela, estado), f"| ~{extras.tokens(janela)} tokens")
afirmar(not any(extras.fatos_na_conversa(janela, estado).values()),
        "na janela de 6 turnos, o pedido e o valor compensado já sumiram")

cliente: Beleza. E o horário de atendimento?
atendente: O atendimento por chat funciona todos os dias, no horário comercial.
cliente: Ok. Posso pagar com vale-refeição?
atendente: Sim, os principais vales aparecem como opção no pagamento.
cliente: Show. Última: o entregador pode deixar na portaria?
atendente: Pode, desde que você marque a opção de entrega na portaria ao fazer o pedido.

fatos na janela: {'pedido_id': False, 'compensado': False} | ~97 tokens


## X2.3 — Resumo contínuo

Troca os turnos antigos por um resumo. O resumo abaixo foi **escrito à mão** no estilo de um resumo genérico de modelo (o resumo real depende do modelo e do prompt de resumo). É fluido e correto, mas sem números: resumos tendem a manter a história e perder os dados.

In [4]:
print(extras.RESUMO_GENERICO)
fatos_resumo = extras.fatos_na_conversa(extras.RESUMO_GENERICO, estado)
print("\nfatos no resumo:", fatos_resumo, f"| ~{extras.tokens(extras.RESUMO_GENERICO)} tokens")
afirmar(not all(fatos_resumo.values()), "este resumo perdeu o número do pedido e o valor já compensado")

O cliente relatou que um pedido chegou incompleto, com itens faltando. A atendente pediu detalhes, resolveu a situação com compensações e o cliente agradeceu. Depois, tirou dúvidas sobre cobertura, troca de endereço, fidelidade, horário e formas de pagamento.

fatos no resumo: {'pedido_id': False, 'compensado': False} | ~64 tokens


## X2.4 — O estado estruturado sobrevive

O estado do atendimento (`session.state`) não faz parte da conversa: comprimir a conversa não o toca. O contexto passa a ser **estado + janela curta**: os dados críticos vêm do código, a janela dá o tom.

In [5]:
enxuto = extras.contexto_com_estado(real.estado, conversa, 4)
print(enxuto)
fatos_enxuto = extras.fatos_na_conversa(enxuto, estado)
print("\nfatos:", fatos_enxuto)
esperar(real, all(fatos_enxuto.values()), "com estado + janela, o pedido e o valor compensado continuam disponíveis")
esperar(real, real.estado.get("pedido_id") == "4822" and real.estado.get("compensacoes") == 18.0,
        "o estado estruturado do atendimento não foi afetado por nenhuma compressão")

from rangoja.notebook import Tabela
mostrar_tabela = Tabela(["estratégia", "~tokens", "pedido", "valor compensado"], [
    ["conversa inteira (20 turnos)", str(extras.tokens(completa)), *map(str, extras.fatos_na_conversa(completa, estado).values())],
    ["janela de 6 turnos", str(extras.tokens(janela)), *map(str, extras.fatos_na_conversa(janela, estado).values())],
    ["resumo genérico", str(extras.tokens(extras.RESUMO_GENERICO)), *map(str, fatos_resumo.values())],
    ["estado + janela de 4", str(extras.tokens(enxuto)), *map(str, fatos_enxuto.values())],
])
mostrar_tabela

ESTADO: {'pedido_id': '4822', 'compensacoes': 18.0}
cliente: Ok. Posso pagar com vale-refeição?
atendente: Sim, os principais vales aparecem como opção no pagamento.
cliente: Show. Última: o entregador pode deixar na portaria?
atendente: Pode, desde que você marque a opção de entrega na portaria ao fazer o pedido.

fatos: {'pedido_id': True, 'compensado': True}


estratégia,~tokens,pedido,valor compensado
conversa inteira (20 turnos),332,True,True
janela de 6 turnos,97,False,False
resumo genérico,64,False,False
estado + janela de 4,78,True,True


É o mesmo argumento de N1.5 (a consulta ao pedido que se repetia a cada mensagem): a verdade do atendimento mora no estado, não na conversa. Compressão boa é a que não precisa ser perfeita.